In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()

while not (ROOT / "spark").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

print("Project root:", ROOT)

Project root: /Users/libowen/github/behavior-aware wind turbine/prediction


## running time: 263m17s

In [2]:
from argparse import Namespace
import pickle
import torch
from torch.utils.data import DataLoader
import random
import numpy as np

from spark.streaming_dataset import TurbineStreamingDataset
from spark.train_utils_streaming import train_streaming, test_streaming

from ml.models.lstm import LSTM


args = Namespace(
    epochs=30,
    lr=0.001,
    batch_size=512,
    device=(
        "cuda" if torch.cuda.is_available()
        else "mps" if torch.backends.mps.is_available()
        else "cpu"
    ),
    num_lags=24,
    seed=0,
    forecast_horizon=3,  # align with federated 3-step task
)


# Load the global y_scaler saved by spark_preprocess.py for inverse-transform metrics
y_scaler_path = ROOT / "spark" / "processed_data" / "global_y_scaler.pkl"
if y_scaler_path.exists():
    with open(y_scaler_path, "rb") as f:
        y_scaler = pickle.load(f)
    print(f"Loaded global y_scaler  data_min={y_scaler.data_min_}  data_max={y_scaler.data_max_}")
else:
    y_scaler = None
    print("WARNING: global_y_scaler.pkl not found -- metrics will be in scaled [0,1] units.")
    print("   Run spark/spark_preprocess.py once to generate the scaler file.")


def set_seed(seed=0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    if torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def main():

    set_seed(args.seed)

    static_cols = ("Capacity_kw", "age")

    train_dataset = TurbineStreamingDataset(
        data_dir = str(ROOT / "spark" / "processed_data" / "train"),
        static_cols=static_cols,
        shuffle=True,
        forecast_horizon=args.forecast_horizon,
    )

    val_dataset = TurbineStreamingDataset(
        data_dir = str(ROOT / "spark" / "processed_data" / "val"),
        static_cols=static_cols,
        shuffle=False,
        forecast_horizon=args.forecast_horizon,
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=args.batch_size,
        num_workers=0,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=args.batch_size,
        num_workers=0,
    )

    sample_x, sample_y = next(iter(train_loader))

    input_dim = sample_x.shape[-1]
    out_dim = sample_y.shape[-1]  # automatically 3 with forecast_horizon=3

    print(f"input_dim={input_dim}  out_dim={out_dim}")

    model = LSTM(
        input_dim=input_dim,
        lstm_hidden_size=128,
        num_lstm_layers=1,
        layer_units=[128],
        num_outputs=out_dim,
        matrix_rep=True,
    )

    device = torch.device(args.device)

    best_model = train_streaming(
        model,
        train_loader,
        val_loader,
        device,
        epochs=args.epochs,
        lr=args.lr,
        y_scaler=y_scaler,  # inverse transform during training logs
    )

    val_metrics = test_streaming(best_model, val_loader, device, y_scaler=y_scaler)

    print("")
    print("Final Validation Metrics (original scale if scaler loaded):")
    mse, rmse, mae, r2, nrmse = val_metrics
    print(f"  MSE={mse:.4f}  RMSE={rmse:.4f}  MAE={mae:.4f}  R2={r2:.4f}  NRMSE={nrmse:.4f}")


if __name__ == "__main__":
    main()


   Run spark/spark_preprocess.py once to generate the scaler file.
input_dim=9  out_dim=3


/Users/libowen/Library/Python/3.10/lib/python/site-packages/torch/nn/modules/rnn.py:82: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn("dropout option adds dropout after all but last "



Epoch 1/30
Train → MSE:0.0058 RMSE:0.0760 MAE:0.0396 R2:0.5892
Val   → MSE:0.0057 RMSE:0.0753 MAE:0.0358 R2:0.6702

Epoch 2/30
Train → MSE:0.0048 RMSE:0.0695 MAE:0.0356 R2:0.6564
Val   → MSE:0.0051 RMSE:0.0712 MAE:0.0362 R2:0.7058

Epoch 3/30
Train → MSE:0.0045 RMSE:0.0668 MAE:0.0338 R2:0.6822
Val   → MSE:0.0065 RMSE:0.0808 MAE:0.0408 R2:0.6206

Epoch 4/30
Train → MSE:0.0043 RMSE:0.0658 MAE:0.0334 R2:0.6921
Val   → MSE:0.0057 RMSE:0.0756 MAE:0.0431 R2:0.6680

Epoch 5/30
Train → MSE:0.0043 RMSE:0.0657 MAE:0.0330 R2:0.6928
Val   → MSE:0.0050 RMSE:0.0708 MAE:0.0368 R2:0.7085

Epoch 6/30
Train → MSE:0.0041 RMSE:0.0644 MAE:0.0322 R2:0.7054
Val   → MSE:0.0052 RMSE:0.0724 MAE:0.0456 R2:0.6954

Epoch 7/30
Train → MSE:0.0041 RMSE:0.0639 MAE:0.0318 R2:0.7095
Val   → MSE:0.0046 RMSE:0.0679 MAE:0.0330 R2:0.7326

Epoch 8/30
Train → MSE:0.0040 RMSE:0.0633 MAE:0.0317 R2:0.7154
Val   → MSE:0.0046 RMSE:0.0675 MAE:0.0343 R2:0.7351

Epoch 9/30
Train → MSE:0.0040 RMSE:0.0631 MAE:0.0313 R2:0.7170
Val   → 